
# 04 风险模型与收益归因：从协方差到 PnL 拆解

> **本章要解决的问题**：组合的风险到底从哪来？为什么样本协方差矩阵不能直接用？VaR 算出来 2%，第二天亏 5% 是不是模型错了？每天赚的钱，有多少是市场给的、多少是因子给的、多少是选股给的、多少被成本吃掉了？

## 本章导读

上一章讲执行，这一章讲“风险账本”和“收益账本”。两份账本是同一个硬币的两面：**风险模型负责解释波动，归因模型负责解释收益。**

按“估计 → 暴露 → 风险 → 尾部 → 归因”的顺序展开：

1. 协方差估计：样本、EWMA、收缩；
2. 因子模型与暴露：行业/市值中性化；
3. 风险预测与分解：因子风险、特质风险、成分风险；
4. VaR、ES 与尾部：三种算法与回测；
5. 压力测试、回撤与杠杆：极端情景下的生存；
6. 每日 PnL 归因：市场、行业、风格、选股、成本。

读的时候带着两个问题：**你的风险真的来自你以为的地方吗？你的收益真的是你赚的吗？** 本章数据全部为合成教学数据，参数为教学示例，不代表任何真实市场。

### 学习目标

- **任务 1**：用样本、EWMA、收缩三种方法估计协方差，并比较它们的条件数与样本外预测误差。
- **任务 2**：用横截面回归估计因子收益与暴露，把信号做行业/市值中性化。
- **任务 3**：把组合风险拆成因子风险与特质风险，并算成分风险贡献。
- **任务 4**：用参数法、历史法、蒙特卡洛法算 VaR/ES，并做超限率回测。
- **任务 5**：跑一个因子冲击的压力情景，并算回撤与杠杆上限。
- **任务 6**：做一张每日 PnL 归因表，让各分项加总等于实际 PnL。

### 先修

第 09 章组合优化；第 26 章组合风险；第 30 章收益归因；第 34 章均值方差与风险平价；会 numpy/pandas 与基础统计。

---

## 在量化研究中的位置

- **谁用**：组合经理、风险/归因研究员、FOF、风控与合规。
- **输入**：收益面板、暴露（行业/风格）、组合权重、成本与交易记录。
- **输出**：协方差估计、风险分解、VaR/ES、压力损失、归因表。
- **在流程的哪一步**：信号 → 组合优化 → **风险预算与归因** → 执行；前面是 34 的优化，后面是 03 的执行。
- **失败模式**：直接用样本协方差、VaR 当最大损失、压力情景拍脑袋、归因加总对不上、只看总收益不看成本。
- **最小工作流**：估协方差 → 估暴露 → 风险分解 → 尾部/压力 → 归因对账。

---

### 环境准备



In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



---

## 1. 协方差估计：样本、EWMA、收缩

### 1.1 样本协方差的三个毛病

设 $N$ 个资产、$T$ 期收益，样本协方差 $\hat\Sigma$ 是 $N\times N$ 矩阵。它的毛病：

1. **维度灾难**：$N$ 接近甚至超过 $T$ 时，$\hat\Sigma$ 接近奇异，条件数爆炸；
2. **噪声大**：每个元素都是一个估计，组合起来误差更大；
3. **不稳定**：加一个月数据，权重可能大变。

### 1.2 EWMA：给近期数据更高权重

指数加权协方差给 $t-i$ 期收益权重 $\lambda^{i}$（$0<\lambda<1$），$\lambda$ 越大越平滑、越小越灵敏。常见 $\lambda=0.94$（日频），对应半衰期约 11 天。EWMA 能跟上波动率变化，但维度灾难还在。

### 1.3 收缩：把噪声“压”向一个结构化目标

收缩估计是两者的加权：

$$\hat\Sigma_{\text{shrink}}=(1-\alpha)\hat\Sigma+\alpha\,\text{Target},$$

目标可以是单位阵、对角阵、或单因子模型。$\alpha$ 越大越稳、偏差越大；实务中常用 Ledoit–Wolf 方法自动选 $\alpha$。**收缩牺牲一点无偏性，换来可逆性与稳定性。**

### 1.4 代码：三种估计的对比



In [2]:

import numpy as np

rng = np.random.default_rng(5401)
N, T = 60, 120
B = np.column_stack([1.0 + rng.normal(0, 0.3, N),  # 因子1：市场（beta≈1）
                      rng.normal(0, 0.8, (N, 2))])  # 因子2/3：风格
F = rng.normal(0, 0.01, (T, 3))
eps = rng.normal(0, 0.02, (N, T))
R = F @ B.T + eps.T                             # T×N 收益矩阵（教学数据）
train, test = R[:60], R[60:]


def ewma_cov(X, lam=0.94):
    Xc = X - X.mean(axis=0)
    w = lam ** np.arange(len(X))[::-1]
    w = w / w.sum()
    return (Xc * w[:, None]).T @ Xc


def shrink_cov(S, alpha=0.3):
    return (1 - alpha) * S + alpha * np.diag(np.diag(S))


w = np.ones(N) / N
S_sample = np.cov(train, rowvar=False)
S_ewma = ewma_cov(train)
S_shrink = shrink_cov(S_ewma)
actual_var = float(w @ np.cov(test, rowvar=False) @ w)

def min_var(S, ridge=0.01):
    """带惩罚的最小方差权重：w ∝ (S + ridge·tr(S)/N·I)^{-1} 1"""
    Sinv = np.linalg.inv(S + ridge * np.trace(S) / N * np.eye(N))
    ww = Sinv @ np.ones(N)
    return ww / ww.sum()


print(f"{'方法':<10}{'条件数':>10}{'等权预测方差':>14}{'等权实际/预测':>13}"
      f"{'最小方差杠杆':>13}{'最小方差实际/预测':>15}")
for name, S in [("样本", S_sample), ("EWMA", S_ewma), ("EWMA+收缩", S_shrink)]:
    pred_eq = float(w @ S @ w)
    w_mv = min_var(S)
    pred_mv = float(w_mv @ S @ w_mv)
    real_mv = float(w_mv @ np.cov(test, rowvar=False) @ w_mv)
    print(f"{name:<10}{np.linalg.cond(S):>10,.0f}{pred_eq:>14.3e}{actual_var / pred_eq:>13.2f}"
          f"{np.abs(w_mv).sum():>13.2f}{real_mv / pred_mv:>15,.1f}")


方法               条件数        等权预测方差      等权实际/预测       最小方差杠杆      最小方差实际/预测
样本        29,872,734,442,527,488     1.075e-04         1.19         3.91          218.3
EWMA      36,952,623,822,809,752     9.839e-05         1.30         3.64          237.0
EWMA+收缩           45     7.183e-05         1.78         1.75            4.0



**读数与应用**：样本协方差在 $N=60$、$T=60$ 时接近奇异，条件数 $3\times10^{16}$；EWMA 病态依旧；收缩之后条件数降到 45。**最刺眼的是最小方差那一列**：用样本/EWMA 估的协方差做优化，组合杠杆 3.6–3.9 倍，样本外实际方差却是预测的 218–237 倍——模型在样本内“看起来”风险很小，样本外被放大两个数量级；收缩后杠杆降到 1.75 倍、误差 4 倍。**协方差的病，在等权组合上看不出来，在优化组合上会被放大。**

**停下来想一想**：同样一份协方差，为什么等权组合的预测误差不算大，最小方差组合却差两个数量级？

<details><summary>对照你的思路</summary>

因为优化会**主动寻找“样本内方差最小”的方向**，而这些方向往往正是估计误差最大的方向（特征值接近 0 的方向）。等权组合不挑方向，所以对误差不敏感；优化组合专挑误差大的方向下注，于是样本内很漂亮、样本外崩掉。这就是“优化对输入误差高度敏感”的典型例子。

</details>

**常见错误**：

1. 直接用样本协方差做优化，权重被噪声主导。
2. EWMA 的 $\lambda$ 用样本内最优，样本外失效。
3. 收缩只往单位阵缩，忽略资产之间的真实相关性。

---

## 2. 因子模型与暴露：收益从哪来

### 2.1 因子模型

把收益写成“共同因子 + 特质”：

$$r_i=X_i f+u_i,$$

其中 $X_i$ 是资产 $i$ 的因子暴露（行业哑变量、市值、动量等），$f$ 是因子收益，$u_i$ 是特质收益。全市场写成矩阵就是 $r=Xf+u$。

### 2.2 暴露估计：横截面回归

给定某天的收益和暴露矩阵，用最小二乘估当天因子收益：

$$\hat f=(X^\top X)^{-1}X^\top r.$$

估出来的 $\hat f$ 可以理解为“当天行业/风格各贡献了多少收益”，是**组合层面的因子收益**，不是某只股票涨跌的原因。

### 2.3 行业/市值中性化

想让信号不押注行业和市值，就把信号对行业哑变量和市值回归，取残差：

$$\tilde s=s-Z\hat\gamma.$$

残差与 $Z$ 正交，意味着组合在这些维度上的暴露为 0。**中性化只去掉你放进去的因子，没放进去的暴露还在。**

### 2.4 代码：暴露估计与中性化



In [3]:

import numpy as np

rng = np.random.default_rng(5402)
N = 300
industry = rng.integers(0, 3, N)
dummies = np.eye(3)[industry]
size = rng.normal(0, 1, N)
mom = 0.3 * size + rng.normal(0, 0.8, N)

X = np.column_stack([dummies, size, mom])
f_true = np.array([0.0020, -0.0010, 0.0015, 0.0040, -0.0020])
ret = X @ f_true + rng.normal(0, 0.015, N)

f_hat, *_ = np.linalg.lstsq(X, ret, rcond=None)
print("估计因子收益:", np.round(f_hat, 4))
print("真实因子收益:", f_true)

signal = 0.6 * size + 0.4 * mom + rng.normal(0, 1, N)
Z = np.column_stack([dummies, size])
gamma, *_ = np.linalg.lstsq(Z, signal, rcond=None)
neutral = signal - Z @ gamma

print(f"原始信号：行业均值 {np.round([signal[industry == i].mean() for i in range(3)], 2)}，"
      f"与 size 相关 {np.corrcoef(signal, size)[0, 1]:.2f}")
print(f"中性化后：行业均值 {np.round([neutral[industry == i].mean() for i in range(3)], 2)}，"
      f"与 size 相关 {np.corrcoef(neutral, size)[0, 1]:.2f}")

k = 30
w = np.zeros(N)
w[np.argsort(neutral)[-k:]] = 1 / k
w[np.argsort(neutral)[:k]] = -1 / k
print(f"多空组合暴露：行业 {np.round(dummies.T @ w, 3)}，"
      f"size {size @ w:+.3f}，mom {mom @ w:+.3f}，净暴露 {w.sum():+.2f}")


估计因子收益: [ 0.0017 -0.0018  0.0007  0.0047 -0.0014]
真实因子收益: [ 0.002  -0.001   0.0015  0.004  -0.002 ]
原始信号：行业均值 [-0.21  0.14  0.04]，与 size 相关 0.53
中性化后：行业均值 [ 0.  0. -0.]，与 size 相关 -0.00
多空组合暴露：行业 [ 0.167 -0.1   -0.067]，size +0.164，mom +0.907，净暴露 +0.00



**读数与应用**：横截面回归估出的因子收益与真实值很接近（有噪声）；中性化之后，信号的行业均值归零、与市值相关归零——这不是“碰巧”，而是回归残差正交性保证的。但**动量暴露还在**：多空组合 mom 暴露约 +0.9，因为中性化时没有放进动量。**“看似中性、其实还有风格暴露”是归因里最常见的翻车点。**

**停下来想一想**：把信号对行业和市值中性化之后，组合就真的“没有风格暴露”了吗？

<details><summary>对照你的思路</summary>

不是。中性化只去掉回归里放进去的那几个变量；没放进去的风格（动量、波动率、估值、成长）仍然可能暴露。而且中性化是对**信号**做的，组合权重还有截断、整数手、多空约束等操作，可能重新引入暴露。正确做法是中性化后回头检查组合的完整暴露表。

</details>

**常见错误**：

1. 中性化只做市值，忘了行业；或者反过来。
2. 中性化后不回头检查暴露，以为“回归过就干净了”。
3. 用全样本估计暴露，再回测（未来函数）。

---

## 3. 风险预测与分解

### 3.1 组合波动

$$\sigma_p=\sqrt{w^\top\Sigma w}.$$

但只报一个总波动不够。风险来自哪里？是押注了行业，还是选股太集中？

### 3.2 因子风险与特质风险

用因子模型分解：

$$\sigma_p^2=\underbrace{w^\top B\Sigma_F B^\top w}_{\text{因子风险}}+\underbrace{\sum_i w_i^2\sigma_{u,i}^2}_{\text{特质风险}}.$$

因子风险又可按因子拆成 $f_j\cdot(\Sigma_F f)_j$，其中 $f=B^\top w$ 是组合的因子暴露。**把风险拆开，才知道该对冲什么。** 特质风险则分散不掉：每个持仓的特质波动只能靠“多拿几只”摊薄，$w_i^2$ 的平方项让集中持仓的代价很高。

### 3.3 成分风险与边际风险

- 边际风险 $\text{MCTR}_i=\partial\sigma_p/\partial w_i=(\Sigma w)_i/\sigma_p$；
- 成分风险 $\text{CTR}_i=w_i\cdot\text{MCTR}_i$，满足 $\sum_i\text{CTR}_i=\sigma_p$。

这一组数告诉你“哪个持仓最占风险预算”，是风险平价和第 26 章的核心工具。

### 3.4 代码：组合风险分解



In [4]:

import numpy as np

Sigma_F = np.cov(F, rowvar=False)
spec_var = eps.var(axis=1)
N_assets = B.shape[0]
w = np.ones(N_assets) / N_assets

factor_var = float(w @ B @ Sigma_F @ B.T @ w)
spec_risk = float(np.sum(w ** 2 * spec_var))
total_var = factor_var + spec_risk
print(f"组合日波动 {np.sqrt(total_var):.2%}；因子风险占比 {factor_var / total_var:.0%}，"
      f"特质风险占比 {spec_risk / total_var:.0%}")

f_exp = B.T @ w
contrib = f_exp * (Sigma_F @ f_exp)
for j, name in enumerate(["因子1", "因子2", "因子3"]):
    print(f"{name}: 暴露 {f_exp[j]:+.3f}，风险贡献 {contrib[j]:+.3e}"
          f"（{contrib[j] / total_var:+.1%}）")
print(f"因子贡献之和 {contrib.sum():.3e}，因子方差 {factor_var:.3e}")

Sigma_total = B @ Sigma_F @ B.T + np.diag(spec_var)
ctr = w * (Sigma_total @ w) / np.sqrt(total_var)
print("成分风险最大的 5 个持仓:", np.round(np.sort(ctr)[::-1][:5], 4))
print(f"成分风险之和 {ctr.sum():.4f}，应等于组合波动 {np.sqrt(total_var):.4f}")


组合日波动 1.07%；因子风险占比 94%，特质风险占比 6%
因子1: 暴露 +1.006，风险贡献 +1.053e-04（+91.6%）
因子2: 暴露 -0.111，风险贡献 +2.702e-06（+2.3%）
因子3: 暴露 -0.030，风险贡献 +2.683e-07（+0.2%）
因子贡献之和 1.083e-04，因子方差 1.083e-04
成分风险最大的 5 个持仓: [0.0003 0.0003 0.0003 0.0002 0.0002]
成分风险之和 0.0107，应等于组合波动 0.0107



**读数与应用**：等权组合日波动 1.07%，**94% 来自因子、只有 6% 来自特质**；市场因子暴露约 1.0，独占 91.6% 的风险贡献——这个“分散化”组合其实主要是一张市场 beta。因子贡献之和严格等于因子方差（$1.083\times10^{-4}$），成分风险之和等于组合波动（0.0107），两处对账都通过。**风险分解的用处不是“算得更准”，而是“知道钱押在哪”。**

**停下来想一想**：一个 60 只股票的等权组合，风险 94% 来自市场因子，这算“分散”吗？

<details><summary>对照你的思路</summary>

算“分散了个股”，不算“分散了风险”。个股特质风险被摊薄到 6%，但市场 beta 加起来还是 1，大盘跌它照样跌。真正的分散是跨资产、跨因子、跨策略，而不是在一堆高相关股票里平均分配。

</details>

**常见错误**：

1. 只报总波动，不报分解。
2. 把因子贡献的绝对值相加，忽略正负抵消。
3. 用样本协方差做成分风险，符号和量级都可能失真。

---

## 4. VaR、ES 与尾部

### 4.1 三个概念

- **VaR（在险价值）**：给定置信水平 $\alpha$，未来一段时间内的最大可能损失（分位数）；
- **ES（期望损失）**：超过 VaR 的平均损失，也叫 CVaR；
- **回测超限率**：实际损失超过 VaR 的频率，应该接近 $\alpha$。

### 4.2 三种算法

1. **参数法**：假设正态，$\text{VaR}_\alpha=-(\mu+z_\alpha\sigma)$，快但低估尾部；
2. **历史法**：用历史收益的分位数，简单、不假设分布，但依赖样本窗口；
3. **蒙特卡洛**：用 $t$ 分布等厚尾分布模拟，能刻画尾部但依赖分布假设。\n\n三种方法的差异主要来自**对尾部的假设**：正态假设下 ES 与 VaR 的比值固定（约 1.25 倍），厚尾分布下 ES 会明显更大；历史法只看样本里出现过的最坏几天，样本短就低估。

### 4.3 代码：三种 VaR/ES 与压力情景



In [5]:

import numpy as np
from scipy.stats import norm

r_p = R @ w
mu, sd = r_p.mean(), r_p.std(ddof=1)
alpha = 0.05
z = norm.ppf(alpha)

var_param = -(mu + z * sd)
es_param = -(mu - sd * norm.pdf(z) / alpha)
var_hist = -np.percentile(r_p, alpha * 100)
es_hist = -r_p[r_p <= -var_hist].mean()

rng = np.random.default_rng(5404)
nu = 4
sims = mu + sd * np.sqrt((nu - 2) / nu) * rng.standard_t(nu, 200000)
var_mc = -np.percentile(sims, alpha * 100)
es_mc = -sims[sims <= -var_mc].mean()

print(f"{'方法':<8}{'95% VaR':>10}{'95% ES':>10}")
for name, v, e in [("参数法", var_param, es_param),
                   ("历史法", var_hist, es_hist),
                   ("MC-t", var_mc, es_mc)]:
    print(f"{name:<8}{v:>10.2%}{e:>10.2%}")

exceed = int((r_p < -var_param).sum())
print(f"参数法 VaR 超限 {exceed} 天 / {len(r_p)} 天（理论约 {len(r_p) * alpha:.0f} 天）")

scenario = np.array([-3 * F[:, 0].std(), -2 * F[:, 1].std(), 0.0])
stress = float(w @ B @ scenario)
print(f"压力情景（因子1 −3σ、因子2 −2σ）组合损失 {-stress:.2%}")

nav = np.cumprod(1 + r_p)
dd = nav / np.maximum.accumulate(nav) - 1
longest, cur = 0, 0
for x in dd:
    cur = cur + 1 if x < 0 else 0
    longest = max(longest, cur)
print(f"最大回撤 {dd.min():.2%}，最长回撤持续 {longest} 天")
print(f"2% VaR 预算下杠杆上限 {0.02 / var_hist:.2f} 倍（保证金约束另计）")


方法         95% VaR    95% ES
参数法          1.79%     2.24%
历史法          1.93%     2.39%
MC-t         1.63%     2.44%
参数法 VaR 超限 8 天 / 120 天（理论约 6 天）
压力情景（因子1 −3σ、因子2 −2σ）组合损失 2.82%
最大回撤 -11.49%，最长回撤持续 110 天
2% VaR 预算下杠杆上限 1.04 倍（保证金约束另计）



**读数与应用**：本例三种方法的 VaR 都在 1.6%–1.9%、ES 在 2.2%–2.4%；参数法 VaR 在 120 天里超限 8 天（理论约 6 天），样本小、波动聚集时超限会成串出现。压力情景损失 2.82%，是日常 VaR 的 1.5 倍以上；最大回撤 −11.5%、持续 110 天——**回撤持续期往往比幅度更折磨人。** 2% VaR 预算下杠杆上限约 1.04 倍。

**常见错误**：

1. 把 VaR 当最大损失——它只是分位数，5% 的日子会比它更差。
2. 用短窗口历史法，尾部样本只有几天。
3. 超限率高就调大 VaR，而不是检查模型假设。

---

## 5. 压力测试、回撤与杠杆

### 5.1 压力测试的三种做法

- **历史情景**：重放某次危机（如 2015、2020）的因子收益；
- **假设情景**：指定因子冲击（±3σ、行业 −10%）；
- **相关性冲击**：平时相关性 0.3，危机时变 0.8，分散化失效。

压力测试的重点不是“算一个数”，而是**找出组合最脆弱的暴露**：哪类因子同步下跌时，组合会亏得最多。设计情景的顺序是“先看暴露表、再设计冲击”，而不是反过来——你的组合押了什么，就测什么。

### 5.2 回撤与杠杆

- **最大回撤**：净值从峰值到谷底的最大跌幅，比波动更贴近投资者感受；
- **回撤持续期**：从峰值回到前高用了多久；
- **杠杆**：在 VaR 预算、保证金、集中度约束下的最大值，通常是三者最小值。

杠杆不是“能开多大就开多大”：本例 VaR 预算 2%、95% 日 VaR 约 1.9%，杠杆上限约 1.04 倍；再叠加保证金、集中度、涨跌停约束，实际可用杠杆通常是三者最小值。**波动率上升时 VaR 变大，杠杆要自动降下来——这就是风险目标制。**

### 5.3 风险预算怎么用

把风险预算分给因子/策略/持仓，而不是分资金：**风险平价分的是风险贡献，不是权重。** 当某个因子的成分风险超出预算时，要么降暴露，要么加对冲。

---

## 6. 每日 PnL 归因：把收益拆开

### 6.1 归因框架

一天的组合 PnL 可以拆成：

$$\text{PnL}=\underbrace{\sum_j f_j\,(w^\top X_{\cdot j})}_{\text{因子}}+\underbrace{w^\top u}_{\text{选股}}+\underbrace{(-\text{成本})}_{\text{交易成本}}.$$

因子项再细分为市场、行业、风格；选股项是特质收益；成本项按换手×费率。**归因的第一条纪律：各分项加总必须等于实际 PnL，对不上就是有项漏了。**

### 6.2 横截面归因 vs 时间序列归因

- **横截面**：解释“今天为什么跑赢/跑输基准”，看暴露与因子收益；
- **时间序列**：解释“这个组合长期收益来自哪里”，看因子收益的历史平均。

两者不能混：横截面归因每天的残差可以很大，但长期应接近 0。

### 6.3 代码：每日 PnL 归因表



In [6]:

import numpy as np

rng = np.random.default_rng(5406)
N, days = 200, 20
industry = rng.integers(0, 3, N)
dummies = np.eye(3)[industry]
beta = rng.normal(1.0, 0.3, N)
size = rng.normal(0, 1, N)
mom = rng.normal(0, 1, N)
X = np.column_stack([beta, dummies, size, mom])          # 市场 + 3 行业 + 2 风格

signal = 0.5 * size - 0.4 * mom + rng.normal(0, 1, N)
k = 20
w = np.zeros(N)
w[np.argsort(signal)[-k:]] = 1 / k
w[np.argsort(signal)[:k]] = -1 / k

f_days = np.column_stack([
    rng.normal(0.0002, 0.008, days),
    rng.normal(0.0001, 0.005, days),
    rng.normal(0.0000, 0.005, days),
    rng.normal(0.0001, 0.004, days),
    rng.normal(0.0001, 0.006, days),
    rng.normal(0.0000, 0.006, days),
])
eps = rng.normal(0, 0.012, (days, N))
R_p = f_days @ X.T + eps

exposure = w @ X
factor_pnl = f_days * exposure
specific_pnl = w @ eps.T
cost_bps, turnover = 1.5, 0.20
cost = np.full(days, turnover * cost_bps / 1e4)
actual = R_p @ w
net = actual - cost

names = ["市场", "行业1", "行业2", "行业3", "size", "mom"]
print(f"{'分项':<8}{'每日平均PnL':>12}{'累计PnL':>12}")
for j, name in enumerate(names):
    print(f"{name:<8}{factor_pnl[:, j].mean() * 1e4:>10.2f}bp{factor_pnl[:, j].sum() * 1e4:>10.1f}bp")
print(f"{'选股':<8}{specific_pnl.mean() * 1e4:>10.2f}bp{specific_pnl.sum() * 1e4:>10.1f}bp")
print(f"{'成本':<8}{-cost.mean() * 1e4:>10.2f}bp{-cost.sum() * 1e4:>10.1f}bp")
print(f"{'净PnL':<8}{net.mean() * 1e4:>10.2f}bp{net.sum() * 1e4:>10.1f}bp")

resid = actual - (factor_pnl.sum(axis=1) + specific_pnl)
print(f"归因对账残差（平均）{resid.mean():+.2e}，应为 0")
print(f"组合暴露：beta {exposure[0]:+.2f}，行业 {np.round(exposure[1:4], 2)}，"
      f"size {exposure[4]:+.2f}，mom {exposure[5]:+.2f}")


分项           每日平均PnL       累计PnL
市场           -0.55bp     -11.1bp
行业1           0.25bp       5.0bp
行业2           0.72bp      14.3bp
行业3          -1.91bp     -38.1bp
size         28.62bp     572.4bp
mom          -5.87bp    -117.5bp
选股            0.47bp       9.5bp
成本           -0.30bp      -6.0bp
净PnL         21.43bp     428.6bp
归因对账残差（平均）-6.94e-19，应为 0
组合暴露：beta -0.03，行业 [-0.1   0.25 -0.15]，size +1.52，mom -0.99



**读数与应用**：归因表把每日 PnL 拆成市场、行业、风格、选股、成本五项，累计加总与净 PnL 一致，对账残差在 $10^{-19}$ 量级。组合 beta 约 0、行业暴露接近 0，但 size 暴露 +1.52、mom 暴露 −0.99——所以 size 一项贡献 28.6bp/天，而真正的“选股”只有 0.47bp/天，成本 −0.30bp/天。**归因的价值就是戳穿“我以为我在选股”：你可能只是在押 size。** 成本项单日看着小，年化下来约 4%，足以吃掉选股收益。

**停下来想一想**：归因表各项加总正好等于实际 PnL，是不是就说明归因“做对了”？

<details><summary>对照你的思路</summary>

加总相等只是必要条件，不是充分条件。它保证没有漏项，但分项的**归因方式**可能不合理：比如用错误的因子定义、暴露滞后一天、成本摊到错误日期，都会让“加总对得上、分项没意义”。归因还要看稳定性、与暴露表是否一致、长期残差是否接近 0。

</details>

**常见错误**：

1. 归因分项加总不等于实际 PnL，还硬说“差不多”。
2. 只做横截面归因，长期平均残差不为 0 却不深究。
3. 成本只算手续费，不算冲击与滑点（见第 03 章）。

---

## 本章小结

### 一张表

| 部件 | 输入 | 输出 | 典型失败 |
|---|---|---|---|
| 协方差估计 | 收益面板 | Σ（样本/EWMA/收缩） | 直接使用样本矩阵 |
| 因子模型 | 收益 + 暴露 | 因子收益、特质收益 | 暴露用全样本估 |
| 中性化 | 信号 + 行业/市值 | 中性信号 | 漏掉风格暴露 |
| 风险分解 | 权重 + Σ | 因子/特质/成分风险 | 只看总波动 |
| VaR/ES | 收益序列 | 尾部损失 | 把 VaR 当最大损失 |
| 压力测试 | 情景冲击 | 极端损失 | 情景拍脑袋 |
| PnL 归因 | 权重 + 因子收益 | 分项 PnL | 加总对不上 |

### 八句话

1. 样本协方差在维度高时接近奇异，不能直接用。
2. EWMA 跟上波动率变化，但救不了病态矩阵。
3. 收缩牺牲无偏性，换可逆与稳定。
4. 中性化只去掉你放进去的因子。
5. 风险要拆成因子与特质，才知道该对冲什么。
6. VaR 是分位数，不是最大损失；ES 更接近尾部体验。
7. 压力测试管“坏日子”，相关性冲击常被忽略。
8. 归因的第一条纪律是加总等于实际 PnL。

---

## 本章习题

**1.** 为什么 $N=60$、$T=60$ 的样本协方差矩阵不能直接拿去做组合优化？

**2.** EWMA 的 $\lambda=0.94$ 对应多长的半衰期（日频）？$\lambda$ 变小时协方差会怎样？

**3.** 收缩估计 $\hat\Sigma=(1-\alpha)\hat\Sigma_{\text{sample}}+\alpha\,\text{Target}$ 中，$\alpha$ 变大有什么利弊？

**4.** 把信号对行业哑变量和市值回归后取残差，这个残差在行业和市值上的暴露是多少？还可能在哪些风格上有暴露？

**5.** 组合日波动 2%，因子风险占 70%，特质风险占 30%。把因子暴露对冲掉后，波动大约变成多少？

**6.** 95% 日 VaR 为 2%、ES 为 3%。明天亏 6% 算不算“模型错了”？超限率应该接近多少？

**7.** 压力情景：市场因子 −3σ（σ=2%）、行业因子 −2σ（σ=1%），组合市场暴露 0.5、行业暴露 0.2。粗略估计组合损失。

**8.** 归因表里各项加总比实际 PnL 少了 0.3bp/天，可能是什么原因？怎么排查？

<details><summary>参考答案</summary>

**1.** $T=N$ 时矩阵接近奇异、条件数极大，最小特征值接近 0，求逆会放大噪声，优化出的权重极端且不稳定。

**2.** 半衰期 $=\ln 0.5/\ln 0.94\approx11$ 天。$\lambda$ 变小，近期数据权重更大，协方差更灵敏、更抖。

**3.** $\alpha$ 大：矩阵更稳定、可逆，但偏差大、低估真实相关性；$\alpha$ 小：更贴近样本，但噪声和病态风险回来。

**4.** 行业和市值上的暴露为 0（残差与回归元正交）；动量、波动率、估值等没放进去的风格仍可能有暴露。

**5.** 只剩特质风险：$\sigma=\sqrt{0.3}\times2\%\approx1.10\%$。

**6.** 5% 的日子里损失会超过 VaR，亏 6% 并不违反 95% VaR（还可能属于 5% 尾部）；超限率应接近 5%，但真实数据会因肥尾/波动聚集而偏离。

**7.** 市场项 $0.5\times(-3\times2\%)=-3\%$，行业项 $0.2\times(-2\times1\%)=-0.4\%$，合计约 −3.4%（忽略交叉项）。

**8.** 常见原因：成本算漏（冲击/滑点）、因子暴露未更新、特质项没算全、权重与收益日期错位。排查：先核对各分项与原始数据，再逐日对账找跳变点。

</details>

---

## 参考资料

- Ledoit, O. & Wolf, M. (2004). *Honey, I Shrunk the Sample Covariance Matrix*.
- Grinold, R. C. & Kahn, R. N. *Active Portfolio Management*（风险模型与归因）。
- Menchero, J., Orr, D. J., & Wang, J. *The Brinson Model and Factor Models*（归因方法）。
- 第 09、26、30、34 章；第 03 章（执行成本）。
